In [2]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
import pandas as pd

In [33]:
# Collect and parse first page
url = 'https://www.ebay.com/sch/i.html?_from=R40&_nkw=prizm+silver+psa10+basketball&_sacat=0&_sop=10&_ipg=200&_pgn='
pageNum = 1
data = defaultdict(list)

In [34]:
for pageNum in range(1, 100):
    page = requests.get(url+str(pageNum))
    soup = BeautifulSoup(page.text, 'html.parser')
    items = soup.findAll(class_='s-item__details clearfix')
    print(r"Page "+str(pageNum)+r": "+str(len(items)))
    for item in items:
        time = item.find(class_='s-item__listingDate').contents[0].contents[0]
        data["Time"].append(time)
        price = item.find(class_='s-item__price').contents[0]
        try:
            price = price.contents[0]
        except BaseException:
            pass
        price = float(str(price).replace("$", "").replace(",", ""))
        data["Price"].append(price)

Page 1: 202
Page 2: 200
Page 3: 200
Page 4: 200
Page 5: 200
Page 6: 200
Page 7: 200
Page 8: 200
Page 9: 200
Page 10: 200
Page 11: 200
Page 12: 80
Page 13: 79
Page 14: 77
Page 15: 79
Page 16: 78
Page 17: 79
Page 18: 80
Page 19: 80
Page 20: 80
Page 21: 80
Page 22: 78
Page 23: 78
Page 24: 78
Page 25: 74
Page 26: 79
Page 27: 80
Page 28: 77
Page 29: 79
Page 30: 79
Page 31: 79
Page 32: 79
Page 33: 80
Page 34: 79
Page 35: 79
Page 36: 78
Page 37: 79
Page 38: 79
Page 39: 79
Page 40: 79
Page 41: 79
Page 42: 78
Page 43: 79
Page 44: 79
Page 45: 79
Page 46: 78
Page 47: 73
Page 48: 79
Page 49: 78
Page 50: 78
Page 51: 79
Page 52: 79
Page 53: 79
Page 54: 77
Page 55: 79
Page 56: 79
Page 57: 76
Page 58: 79
Page 59: 79
Page 60: 78
Page 61: 79
Page 62: 79
Page 63: 77
Page 64: 79
Page 65: 76
Page 66: 78
Page 67: 79
Page 68: 79
Page 69: 79
Page 70: 74
Page 71: 77
Page 72: 78
Page 73: 69
Page 74: 78
Page 75: 77
Page 76: 74
Page 77: 0
Page 78: 0
Page 79: 0
Page 80: 0
Page 81: 0
Page 82: 0
Page 83: 0
Page 84: 

----

In [137]:
raw = pd.DataFrame(data)
raw.head()

,Price,Time
0,75.00,Mar-6 07:55
1,699.00,Mar-6 07:49
2,199.99,Mar-3 17:45
3,2499.99,Jan-4 11:44
4,899.99,Mar-6 07:47


In [144]:
raw.set_index("Time").to_csv(r"raw_sports_cards.csv")

In [145]:
month_dict = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6, 
              "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
year = 2020
prev_month = 3

def cleanup_date(raw_date):
    
    global month_dict, year, prev_month
    
    raw = str(raw_date)
    date, time = raw.split(' ')
    month, day = date.split('-')
    hour, minute = time.split(':')
    
    month = month_dict[month]
    
    if month == 12 and prev_month != 12:
        year -= 1
    try:
        result = pd.Period(month=month, year=year, freq='T',
                           day=int(day), hour=int(hour), minute=int(minute))
    except BaseException as e:
        print(e)
        print(raw)
        
    prev_month = month
    return result

In [146]:
raw["Time"] = raw["Time"].apply(cleanup_date)

In [150]:
raw.set_index("Time").to_csv(r"clean_sports_cards.csv")